# Image Difference Analysis

This notebook compares the **Golden** reference image with the **Actual** output from the `e2e_raw_pipeline` test to help diagnose processing shifts.

### Setup
Ensure you have `numpy`, `Pillow`, and `matplotlib` installed:
```bash
pip install numpy Pillow matplotlib
```

In [1]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Define paths relative to the repository root (assuming notebook is in tests/e2e/)
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
golden_path = os.path.join(repo_root, "tests", "e2e", "golden", "golden_raw_pipeline.jpg")
actual_path = os.path.join(repo_root, "tests", "data", "raw", "_DSC0355-output.jpg")

def load_image(path):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}")
        return None
    return np.array(Image.open(path).convert('RGB'))

golden = load_image(golden_path)
actual = load_image(actual_path)

if golden is not None and actual is not None:
    print(f"Golden resolution: {golden.shape[1]}x{golden.shape[0]}")
    print(f"Actual resolution: {actual.shape[1]}x{actual.shape[0]}")
    
    if golden.shape != actual.shape:
        print("WARNING: Image dimensions do not match!")

OSError: image file is truncated (62 bytes not processed)

### Calculation of Differences

In [ ]:
if golden is not None and actual is not None and golden.shape == actual.shape:
    # Calculate absolute difference per pixel/channel
    diff = np.abs(golden.astype(np.int16) - actual.astype(np.int16))
    
    # Total difference per pixel (sum of RGB channels)
    diff_sum = np.sum(diff, axis=2)
    
    # Metrics
    mae = np.mean(diff)
    mse = np.mean(diff**2)
    rmse = np.sqrt(mse)
    max_diff = np.max(diff)
    
    # Thresholded difference (pixels where difference > threshold)
    threshold = 5
    diff_pixels = np.sum(diff_sum > threshold)
    total_pixels = golden.shape[0] * golden.shape[1]
    
    print(f"Mean Absolute Error (per channel): {mae:.4f}")
    print(f"Root Mean Squared Error: {rmse:.4f}")
    print(f"Max variation in any channel: {max_diff}")
    print(f"Pixels differing by > {threshold} (total magnitude): {diff_pixels} ({diff_pixels/total_pixels*100:.2f}%)")

### Visualization

In [ ]:
if golden is not None and actual is not None:
    fig, axes = plt.subplots(1, 3, figsize=(20, 10))
    
    axes[0].imshow(golden)
    axes[0].set_title("Golden Reference")
    axes[0].axis('off')

    axes[1].imshow(actual)
    axes[1].set_title("Actual Test Output")
    axes[1].axis('off')

    # Difference heatmap (intensified for visibility)
    if golden.shape == actual.shape:
        diff_vis = np.clip(diff_sum * 10, 0, 255).astype(np.uint8)
        axes[2].imshow(diff_vis, cmap='hot')
        axes[2].set_title("Difference Heatmap (10x intensity)")
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

### Error Distribution

In [ ]:
if golden is not None and actual is not None and golden.shape == actual.shape:
    plt.figure(figsize=(10, 5))
    plt.hist(diff.ravel(), bins=64, range=(1, 64), color='red', alpha=0.7)
    plt.title("Distribution of Pixel Differences (excluding identical pixels)")
    plt.xlabel("Absolute Difference Value")
    plt.ylabel("Pixel Count")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.yscale('log') # Use log scale as small differences are usually much more frequent
    plt.show()